# Setup, settings and test data

Every job in this project is run twice over the same Parquet files: once in a single process
using DuckDB, and once through Apache Spark, an engine built to spread work across many
machines. Spark runs here in local mode, on one machine.

The question is not which engine is better. It is:

> **Does this job need distributing at all?**

That question has a measurable answer, and the answer is different for different kinds of work.

### What distribution costs, and what it buys

Splitting a job across machines does two things at once.

It **costs** you something. A function call becomes a scheduled task. Data that would have moved
within memory gets serialised, sent over a network and rebuilt at the other end. A plan has to be
split into stages, handed to a scheduler, tracked and retried.

It **buys** you something. Many machines instead of one: more cores, more memory, more network
links to storage.

On a large enough job that is an excellent trade, which is why Spark exists and why it is the
right answer for a real class of problems.

**This project only sees the cost side.** Everything runs on one machine, so Spark pays the full
price of distributing and collects none of the benefit. That is a deliberate limit, and it is
exactly the situation of a job that never needed more than one machine: it pays the cost and
never collects the payoff, however large the cluster is.

**Two things differ at once, and this cannot separate them.** Spark distributes the work, and it is
also a different engine: a JVM runtime with a scheduler, against a vectorized C++ engine running
in-process. Part of every gap below is coordination and part of it is implementation, and this
setup cannot say how much is which. That is why the tables report a plain ratio rather than
calling it the cost of distribution.

What can be said without overstating it: these jobs ran faster in a single process, and on one
machine the distributed path paid the full price of distributing while collecting none of the
benefit that would justify it.

### The rules

1. Both engines read the same Parquet files from the same folder.
2. Both see the same table names, so most cases send one identical SQL string to both. Where
   that is impossible the case is marked.
3. Both get the same memory and thread count.
4. One untimed warm-up runs first, so a cold JVM is never measured.
5. Spark's startup time is reported but never added to a timing.
6. Both results are compared, because a fast wrong answer is worthless.
7. Every Spark setting changed from its default makes Spark faster. They are all in
   `bench/config.py`.

One methodology note. Every Parquet file here is written by DuckDB, which decides the row group
size and the column statistics that both engines then read. Rewriting the same data with Spark
and re-running showed the substantive queries were unaffected (a sum went from 4.9x to 10.0x, a
group-by from 9.8x to 10.4x), but results that lean on file statistics moved a lot (a min/max
query went from 124x to 27x). Those cases are flagged in the results tables.

Size is set by `BENCH_ROWS`, default 1M in quick mode and 20M in full. 20M is chosen because the
data plus written output and shuffle spill has to fit the runner's disk, and because a single
machine remains a credible choice for a job that size.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from bench import config as C, datagen, engines
import pandas as pd

print(C.summary())


## The settings

Everything that affects a number is in `bench/config.py`. If a setting looks unfair to either
engine, change it there and re-run.

`spark.sql.shuffle.partitions` defaults to 200. On a single machine that schedules 200 tasks
across a handful of cores, so it is set to the core count instead.

`spark.sql.execution.arrow.pyspark.enabled` is off by default. Without it the timings include a
slow row by row handover of results to Python rather than the query itself.

In [ ]:
print("Spark settings in force:\n")
for k, v in C.SPARK_CONF.items():
    print(f"   {k:<48} {v}")


## The test data

Three tables: one large table of events joined to two small lookups.

The `sales` table is deliberately wide (22 columns) so that reading only the columns a query
names is worth something. It is also deliberately dirty. Quality checks against clean data
measure nothing, so the defects below are injected on purpose and listed rather than left to be
discovered.

In [ ]:
print("DEFECTS PLANTED ON PURPOSE\n")
for k, v in datagen.DEFECTS.items():
    print(f"   {v:<32} {k}")
datagen.build(C.MAIN_SIZE, with_extras=True)

print("\nOn disk:")
display(pd.DataFrame(datagen.disk_report()))


### Companion datasets

Alongside the three main tables we build a few extras so the later notebooks have something
realistic to work with:

| Dataset | Why it exists |
|---|---|
| `returns` | A second large table, so notebook 03 can do a real big-to-big shuffle join |
| `sales_v2` | A later snapshot with changes and deletions, for diffing and reconciliation |
| `sales_part` | A partitioned copy, to test skipping folders rather than filtering rows |
| `sales_small_*` | **200 small files**, for the compaction job every team has |
| `sales_*.csv` | A CSV extract, for the CSV-to-Parquet job |

In [ ]:
paths = datagen.paths(C.MAIN_SIZE)
duck  = engines.get_duckdb()
spark = engines.get_spark()
attached = engines.attach(duck, spark, paths)
print("Both engines can now see:", ", ".join(attached))
print()
print(engines.describe())


## A look at the data itself

In [ ]:
print("sales, one row per purchase, 22 columns:")
display(duck.execute("SELECT * FROM sales LIMIT 5").df())

n = C.MAIN_SIZE
print(f"\n{C.human(n)} sales rows is about {n/3000/365:.1f} years of trading "
      f"for a business doing 3,000 orders a day.")

print("\nThe planted defects:")
display(duck.execute("""
SELECT count(*)                                            AS total_rows,
       count(*) - count(customer_id)                       AS null_customer_id,
       count(*) - count(amount)                            AS null_amount,
       sum(CASE WHEN amount < 0 THEN 1 ELSE 0 END)         AS negative_amount
FROM sales
""").df())

print("\nThe same country written several ways, which notebook 05 cleans up:")
display(duck.execute("""
SELECT upper(trim(country)) AS actually_the_same,
       count(DISTINCT country)                    AS spellings,
       string_agg(DISTINCT '[' || country || ']', '  ') AS as_stored,
       sum(n)                                     AS customers
FROM (SELECT country, count(*) AS n FROM customers GROUP BY 1)
GROUP BY 1 ORDER BY 1 LIMIT 5
""").df())
engines.stop_spark()


---

Notebook `01_scan_and_filter` starts the comparison. Each notebook covers one family of work and
saves its results to `results/`; `10_summary` combines them.